In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install mlflow dagshub -q

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import mlflow

In [6]:
train = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
test = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")

y = train["isFraud"]
test_ids = test["TransactionID"]

X = train.drop(columns=["isFraud"])

In [7]:
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(X[num_cols].median())

X[cat_cols] = X[cat_cols].fillna("missing")
test[cat_cols] = test[cat_cols].fillna("missing")

In [8]:
for df in [X, test]:
    df["log_amt"] = np.log1p(df["TransactionAmt"])
    df["null_count"] = df.isnull().sum(axis=1)

/tmp/ipykernel_57/1559358077.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["log_amt"] = np.log1p(df["TransactionAmt"])
/tmp/ipykernel_57/1559358077.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["null_count"] = df.isnull().sum(axis=1)
/tmp/ipykernel_57/1559358077.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, 

In [9]:
full = pd.concat([X, test], axis=0)

full = pd.get_dummies(full, columns=cat_cols, drop_first=True)

X = full.iloc[:len(X)]
test = full.iloc[len(X):]

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X.fillna(0),
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [17]:
model = LogisticRegression(
    max_iter=2000,
    solver="saga",
    class_weight="balanced",
    n_jobs=-1
)

In [18]:
model.fit(X_train, y_train)

KeyboardInterrupt: 